# 🎯 Priority Classifier v2 — Training in VS Code

**Prerequisiti**: aver eseguito `PriorityClassifier_E5_Colab.ipynb` e aver scaricato nella cartella progetto:
- `X_emb_priority_train.npy`
- `X_emb_priority_test.npy`
- `meta_priority_train.csv`
- `meta_priority_test.csv`

### Cosa cambia rispetto alla v1 (ModelTraining.ipynb)
| | v1 | v2 |
|---|---|---|
| Embedding model | paraphrase-multilingual-mpnet-base-v2 | **intfloat/multilingual-e5-base** |
| Prefisso testo | nessuno | **`query: `** (obbligatorio per E5) |
| C ottimale | non tunato | **ricercato via CV** |
| Coerenza con Area | ❌ modello diverso | ✅ stesso modello E5 |

---
## STEP 1 — Caricamento embedding e metadati

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import warnings
import time
warnings.filterwarnings('ignore')

# Carica embedding pre-calcolati su Colab
X_emb_train = np.load('X_emb_priority_train.npy')
X_emb_test  = np.load('X_emb_priority_test.npy')

# Carica metadati (label + feature strutturali)
meta_train = pd.read_csv('meta_priority_train.csv')
meta_test  = pd.read_csv('meta_priority_test.csv')

print(f"Embedding train: {X_emb_train.shape}")
print(f"Embedding test:  {X_emb_test.shape}")
print(f"Meta train:      {meta_train.shape}")
print(f"Meta test:       {meta_test.shape}")

# Verifica allineamento — FONDAMENTALE
assert len(X_emb_train) == len(meta_train), "❌ Embedding e meta train non allineati!"
assert len(X_emb_test)  == len(meta_test),  "❌ Embedding e meta test non allineati!"
print("\n✅ Embedding e metadati allineati")

# Verifica norma (deve essere ~1.0 — conferma che E5 ha normalizzato)
norme = np.linalg.norm(X_emb_train[:5], axis=1)
print(f"\nNorma L2 primi 5 vettori (deve essere ~1.0): {norme}")

print(f"\nColonne metadati: {list(meta_train.columns)}")
print(f"\nDistribuzione priorita_finale — train:")
print(meta_train['priorita_finale'].value_counts().sort_index())
print(f"\nDistribuzione priorita_finale — test:")
print(meta_test['priorita_finale'].value_counts().sort_index())

---
## STEP 2 — Feature engineering strutturali

Affianchiamo agli embedding due feature strutturali già disponibili al momento della predizione:

**`priorita_iniziale_cliente`** (One-Hot Encoded):  
Il cliente sceglie la priorità nel form ZConnect → disponibile in produzione prima ancora di leggere il testo.  
L'analisi dei pesi (v1) mostra che questa feature pesa ~2x rispetto agli embedding.
Questo è corretto e non è data leakage: l'88% delle volte l'operatore non la corregge.

**`has_urgenza`** (booleana):  
1 se il testo contiene: 'urgente', 'immediato', 'il prima possibile', 'critico'.  
43.5% dei ticket con urgenza sono P1 vs 12.2% senza → segnale forte.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

CAT_COLS  = ['priorita_iniziale_cliente']
BOOL_COLS = ['has_urgenza']

# Gestione NaN (non dovrebbero esserci dopo il preprocessing)
for col in CAT_COLS:
    meta_train[col] = meta_train[col].fillna('sconosciuto')
    meta_test[col]  = meta_test[col].fillna('sconosciuto')

# OHE — fit SOLO su train, transform su entrambi
# handle_unknown='ignore': categorie nel test non viste nel train → vettore zero
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(meta_train[CAT_COLS])

X_cat_train = encoder.transform(meta_train[CAT_COLS])
X_cat_test  = encoder.transform(meta_test[CAT_COLS])

# has_urgenza → matrice sparsa
X_bool_train = sp.csr_matrix(meta_train[BOOL_COLS].values.astype(float))
X_bool_test  = sp.csr_matrix(meta_test[BOOL_COLS].values.astype(float))

print(f"Feature categoriche (OHE): {X_cat_train.shape[1]} colonne")
print(f"  Categorie: {encoder.categories_[0].tolist()}")
print(f"Feature booleane:          {X_bool_train.shape[1]} colonna (has_urgenza)")

# Verifica correlazione has_urgenza → priorità (sanity check)
print(f"\n=== Sanity check has_urgenza ===")
urgenza_arr = meta_train['has_urgenza'].values
priority_arr = meta_train['priorita_finale'].values
for p in ['P1', 'P2', 'P3', 'P4']:
    pct_urgenza = urgenza_arr[priority_arr == p].mean() * 100
    print(f"  {p}: {pct_urgenza:.1f}% ha has_urgenza=1")

---
## STEP 3 — Costruzione feature matrix finale

Combiniamo le tre componenti in un'unica matrice sparsa:

```
X = [  embedding E5 (768d)  |  OHE priorita_iniziale (4d)  |  has_urgenza (1d)  ]
      └── segnale testuale ──┘  └──── segnale cliente ─────┘  └── segnale urgenza ┘
```

Totale: **773 feature** per ticket.

In [ ]:
# Converti embedding densi in sparse per hstack
X_emb_train_sp = sp.csr_matrix(X_emb_train)
X_emb_test_sp  = sp.csr_matrix(X_emb_test)

# hstack: unisce colonna per colonna
X_train = sp.hstack([X_emb_train_sp, X_cat_train, X_bool_train])
X_test  = sp.hstack([X_emb_test_sp,  X_cat_test,  X_bool_test])

y_train = meta_train['priorita_finale'].values
y_test  = meta_test['priorita_finale'].values

print(f"Feature matrix train: {X_train.shape}")
print(f"Feature matrix test:  {X_test.shape}")
print(f"  └── embedding:   768 colonne")
print(f"  └── OHE categ.:  {X_cat_train.shape[1]} colonne")
print(f"  └── has_urgenza: 1 colonna")
print(f"  └── TOTALE:      {X_train.shape[1]} colonne")
print(f"\nClassi target: {np.unique(y_train)}")

---
## STEP 4 — Tuning parametro C con cross-validation

Cerchiamo il C ottimale per LinearSVC.  
C regola il tradeoff bias/varianza:
- **C piccolo** (0.01): forte regolarizzazione → può underfitare
- **C grande** (10): poca regolarizzazione → può overfitare su train

Con embedding già L2-normalizzati, i valori ottimali tipici sono tra 0.1 e 5.0.  
Usiamo 3-fold CV (non 5) per velocità — su 46k ticket è già robusto.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score, StratifiedKFold

VALORI_C = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Tuning C — LinearSVC (3-fold CV, macro F1)...")
print(f"{'C':>6}  {'F1 macro':>10}  {'Std':>8}  {'Tempo':>8}")
print("-" * 42)

risultati_c = {}
for c in VALORI_C:
    clf_tmp = LinearSVC(class_weight='balanced', max_iter=3000, C=c, random_state=42)
    t0 = time.time()
    scores = cross_val_score(
        clf_tmp, X_train, y_train,
        cv=cv, scoring='f1_macro', n_jobs=-1
    )
    elapsed = time.time() - t0
    risultati_c[c] = scores.mean()
    print(f"{c:>6.2f}  {scores.mean():>10.4f}  {scores.std():>8.4f}  {elapsed:>7.0f}s")

C_OTTIMALE = max(risultati_c, key=risultati_c.get)
print(f"\n🏆 C ottimale: {C_OTTIMALE} → F1 macro = {risultati_c[C_OTTIMALE]:.4f}")

---
## STEP 5 — Training e valutazione finale

In [ ]:
from sklearn.metrics import classification_report, f1_score

print(f"Training LinearSVC con C={C_OTTIMALE}...")
t0 = time.time()

clf = LinearSVC(
    class_weight='balanced',  # Compensa sbilanciamento P4 (2.5% del dataset)
    max_iter=3000,
    C=C_OTTIMALE,
    random_state=42
)
clf.fit(X_train, y_train)
print(f"✅ Training completato in {time.time()-t0:.1f}s")

y_pred = clf.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average='macro')
accuracy = (y_pred == y_test).mean()

print(f"\n{'='*60}")
print(f"RISULTATI CLASSIFICATORE PRIORITÀ v2 (E5-base + LinearSVC)")
print(f"{'='*60}")
print(f"Macro F1:  {macro_f1:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print()
print(classification_report(y_test, y_pred, digits=3))

---
## STEP 6 — Confronto v1 vs v2 e analisi pesi

Confronto diretto con i risultati della v1 (mpnet + split temporale Nov 2025).

In [ ]:
# Risultati v1 per confronto (mpnet, split temporale, C=1.0 non tunato)
v1_results = {
    'P1': {'precision': 0.630, 'recall': 0.950, 'f1': 0.760},
    'P2': {'precision': 0.870, 'recall': 0.800, 'f1': 0.840},
    'P3': {'precision': 0.990, 'recall': 0.910, 'f1': 0.950},
    'P4': {'precision': 0.970, 'recall': 0.740, 'f1': 0.840},
    'macro_f1': 0.843,
    'accuracy': 0.880
}

from sklearn.metrics import classification_report
import re

print("=" * 65)
print(f"{'Metrica':<30} {'v1 (mpnet)':>12} {'v2 (E5)':>12} {'Delta':>8}")
print("-" * 65)

macro_v2 = f1_score(y_test, y_pred, average='macro')
acc_v2   = (y_pred == y_test).mean()

print(f"{'Macro F1':<30} {v1_results['macro_f1']:>12.3f} {macro_v2:>12.3f} {macro_v2-v1_results['macro_f1']:>+8.3f}")
print(f"{'Accuracy':<30} {v1_results['accuracy']:>12.3f} {acc_v2:>12.3f} {acc_v2-v1_results['accuracy']:>+8.3f}")

from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=['P1','P2','P3','P4'])
for i, classe in enumerate(['P1','P2','P3','P4']):
    f1_v1 = v1_results[classe]['f1']
    print(f"{'F1 '+classe:<30} {f1_v1:>12.3f} {f1[i]:>12.3f} {f1[i]-f1_v1:>+8.3f}")
print("=" * 65)

In [ ]:
# Analisi importanza gruppi di feature (embedding vs categoriche vs urgenza)
# Stesso calcolo della v1 — confrontiamo i valori

n_emb  = 768
n_cat  = X_cat_train.shape[1]
n_bool = 1  # has_urgenza

print("=== Peso medio assoluto per gruppo di feature ===")
print(f"{'Classe':<8} {'Embedding':>12} {'Priorità_cli':>14} {'has_urgenza':>13}")
print("-" * 52)

for i, classe in enumerate(clf.classes_):
    coef = clf.coef_[i]
    peso_emb  = np.abs(coef[:n_emb]).mean()
    peso_cat  = np.abs(coef[n_emb:n_emb+n_cat]).mean()
    peso_bool = np.abs(coef[n_emb+n_cat:]).mean()
    print(f"{classe:<8} {peso_emb:>12.4f} {peso_cat:>14.4f} {peso_bool:>13.4f}")

print()
print("Nota: se 'Embedding' < 'Priorità_cli' significa che il modello")
print("si fida più della scelta del cliente che del testo del ticket.")
print("Questo è atteso (88% dei casi non viene corretto dall'operatore).")

---
## STEP 7 — Confusion matrix e analisi errori

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix assoluta
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    labels=['P1', 'P2', 'P3', 'P4'],
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix — Assoluta')

# Confusion matrix normalizzata per riga (recall per classe)
cm_norm = confusion_matrix(y_test, y_pred, labels=['P1','P2','P3','P4'], normalize='true')
import seaborn as sns
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=['P1','P2','P3','P4'], yticklabels=['P1','P2','P3','P4'],
    ax=axes[1], vmin=0, vmax=1
)
axes[1].set_title('Confusion Matrix — Normalizzata (recall)')
axes[1].set_xlabel('Predizione'); axes[1].set_ylabel('Verità')

plt.suptitle('Priority Classifier v2 — E5-base + LinearSVC', fontsize=13)
plt.tight_layout()
plt.show()

# Errori più frequenti
df_err = pd.DataFrame({'reale': y_test, 'predetto': y_pred})
errori = df_err[df_err['reale'] != df_err['predetto']]
print(f"\nErrori totali: {len(errori):,} su {len(y_test):,} ({len(errori)/len(y_test)*100:.1f}%)")
print("\nTop coppie di confusione:")
print(
    errori.groupby(['reale', 'predetto']).size()
    .sort_values(ascending=False).head(8).to_string()
)

---
## STEP 8 — Salvataggio modello

In [ ]:
import joblib, json, os

os.makedirs('modello_priority_v2', exist_ok=True)

joblib.dump(clf,     'modello_priority_v2/classificatore_svc.pkl')
joblib.dump(encoder, 'modello_priority_v2/ohe_encoder.pkl')

metadata = {
    'versione': 'v2',
    'modello_embedding': 'intfloat/multilingual-e5-base',
    'prefisso_e5': 'query: ',
    'classificatore': 'LinearSVC',
    'C_ottimale': C_OTTIMALE,
    'feature': [
        'embedding_e5_768d',
        'priorita_iniziale_cliente_ohe',
        'has_urgenza'
    ],
    'classi': clf.classes_.tolist(),
    'split': 'temporale',
    'soglia_split': '2025-11-01',
    'macro_f1_test': round(float(f1_score(y_test, y_pred, average='macro')), 4),
    'accuracy_test': round(float((y_pred == y_test).mean()), 4),
    'n_train': int(len(y_train)),
    'n_test': int(len(y_test)),
    'keyword_urgenza': ['urgente', 'immediato', 'il prima possibile', 'critico']
}

with open('modello_priority_v2/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("✅ Modello salvato in 'modello_priority_v2/'")
print(json.dumps(metadata, indent=2, ensure_ascii=False))